# Example: Bigdata.com Knowledge Graph API

Client-facing Knowledge Graph API example on [Bigdata.com](https://bigdata.com) for resolving names to IDs and enriching IDs back to entity metadata.

**What this example shows**
- Resolve company names to entity IDs for search filters
- Resolve entity IDs found in detections back to names and metadata
- Retrieve source metadata for source-level filtering workflows



In [1]:
import requests
import json
from datetime import datetime, timedelta
from print_helpers import (
    print_companies,
    print_entity_details
)

## Setup & Configuration


In [2]:
# Load credentials from .env file
import os
from dotenv import load_dotenv
load_dotenv()

# Bigdata.com API Configuration
API_BASE_URL = "https://api.bigdata.com"
API_KEY = os.getenv("BIGDATA_API_KEY")
if not API_KEY:
    raise ValueError("Set BIGDATA_API_KEY in .env")

# Knowledge Graph API Endpoints
KG_COMPANIES_ENDPOINT = f"{API_BASE_URL}/v1/knowledge-graph/companies"
KG_ENTITIES_ENDPOINT = f"{API_BASE_URL}/v1/knowledge-graph/entities/id"
KG_SOURCES_ENDPOINT = f"{API_BASE_URL}/v1/knowledge-graph/sources"

# Authentication via API key
session = requests.Session()
session.headers.update({"Content-Type": "application/json", "X-API-KEY": API_KEY})
print("✅ API key configured")

✅ API key configured


## Key considerations before starting

The provider APIs do not have full coverage for parameter validation—double-check your request against the [API reference](https://docs.bigdata.com/api-reference) to avoid issues from typos or invalid values.



## 1. Company lookup: name to entity ID

Search and Volume APIs filter by **entity ID** (e.g. `entity.any_of: ["D8442A"]`). To get IDs from company names, use the Knowledge Graph **Companies** endpoint: send a query string (e.g. `"Apple"`) and receive a list of matching companies with their IDs. The results are ordered by a popularity ranking that favors well-known companies, so typically the most prominent or widely-recognized companies appear first. Pick the entry that matches your intent.

**Endpoint:** [Find companies](https://docs.bigdata.com/api-reference/knowledge-graph/find-companies)

**Optional filters** (narrow results by company attributes):
- `types`: e.g. `["PUBLIC"]` (public companies), `["PRIVATE"]` (private companies)
- `countries`: ISO 3166-1 alpha-2 codes (e.g. `["US", "FR"]`)
- `sectors`: e.g. `["Technology", "Financials", "Industrials"]`


In [3]:
company_search_payload = {
    "query": "Apple",
    # Optional: narrow by company attributes
    # "types": ["PUBLIC"],
    # "countries": ["US", "FR"],
    # "sectors": ["Technology", "Financials", "Industrials"]
}

response = session.post(KG_COMPANIES_ENDPOINT, json=company_search_payload)
data = response.json()
print_companies(response, data)

✅ Status: 200
🏢 Found 20 companies

1. Apple Inc.                               (ID: D8442A)
2. Apple Hospitality REIT Inc.              (ID: 9D3360)
3. Apple Finance Ltd.                       (ID: 72BB2B)
4. Apple International Co. Ltd.             (ID: 3ORTET)
5. Apple Patent Law Firm                    (ID: Q15XYP)


['D8442A', '9D3360', '72BB2B', '3ORTET', 'Q15XYP']

You'll see a list of matching companies with their IDs—pick the entry that matches your intent. Try changing the query (e.g. `"Microsoft"`, `"Tesla"`), using additional filters, and re-running the cell above to find other company IDs.


You can also look up companies directly using standard identifiers such as ISIN, CUSIP, SEDOL, or TICKER.
The Knowledge Graph API includes dedicated endpoints for these identifier lookups:
- `/knowledge-graph/companies/isin` — Find a company using its ISIN
- `/knowledge-graph/companies/cusip` — Find a company using its CUSIP
- `/knowledge-graph/companies/listing` — Find a company by ticker or listing code

For example, providing an ISIN or ticker will return the corresponding company entity.


## 2. From Entity ID to details

Next, we resolve **entity IDs to names and metadata**—useful when you have IDs from Search or CoMentions results. When you have a list of entity IDs—such as those returned from Search or CoMentions results (`detections` in chunks)—you can resolve them to names and metadata using the **Entities by ID** endpoint. Simply send up to 100 RavenPack IDs per request and receive a mapping of ID to entity details (name, type, etc.), which is especially useful for labeling entities in your pipelines.

**Endpoint:** [Get entities by ID](https://docs.bigdata.com/api-reference/companies/find-by-details)


In [4]:
entity_lookup_payload = {
    "values": ["D8442A"]
}

response = session.post(KG_ENTITIES_ENDPOINT, json=entity_lookup_payload)
data = response.json()
print(data)

{'results': {'D8442A': {'id': 'D8442A', 'name': 'Apple Inc.', 'description': 'Apple Inc. (formerly Apple Computer Inc.), incorporated on January 03, 1977, designs, manufactures, and markets mobile communication and media devices, personal computing products, and portable digital music players worldwide.', 'type': 'PUBLIC', 'country': 'US', 'sector': 'Technology', 'industry_group': 'Computer Hardware', 'industry': 'Computer Hardware', 'favicon': 'http://www.apple.com/favicon.ico', 'webpage': 'http://www.apple.com', 'isin_values': ['CA03785Y1007', 'CA0379741022', 'TH0150120408', 'TH0241121001', 'TH0809121500', 'TH8483124708', 'US0378331005'], 'cusip_values': ['037833100', '03785Y100', '037974102', 'P0R684385', 'Y100G4352', 'Y49876132', 'Y689DP202', 'Y985KH543'], 'sedol_values': ['2046251', 'BNC31R1', 'BPXWT99', 'BRC1SP0', 'BTTRQ40', 'BVBDDG1', 'BWBYBW3'], 'listing_values': ['XBKK:AAPL80', 'XNAS:AAPL'], 'category': 'companies'}}, 'metadata': {'request_id': '433b9617-0b63-477e-b036-c692280

## 3. Entity discovery workflow

We now tie the Knowledge Graph and Search APIs together in one workflow.

**Why this workflow?** When you search with an entity filter, each chunk includes `detections`: entity IDs mentioned in that text. To see **who** or **what** those are—and to discover which companies, people, or topics are co-mentioned with your focal entity—you resolve those IDs with the Knowledge Graph. This workflow shows that full loop: company name → filtered search → entity IDs from chunks → resolved names.

Steps:
1. **Look up** a company (e.g. Microsoft) to get its entity ID  
2. **Search** with that entity filter (Search API)  
3. **Extract** entity IDs from the first chunk’s `detections`  
4. **Resolve** those IDs to names via the Knowledge Graph  

You can reuse this pattern whenever you need to go from company name → filtered search → labelled entities.


In [5]:
# Find Microsoft Entity ID
# For well-known companies with correct spelling, the first result is typically correct
company_query = {"query": "Microsoft"}
response = session.post(KG_COMPANIES_ENDPOINT, json=company_query)
companies = response.json()['results']

microsoft_id = companies[0]['id']
microsoft_name = companies[0]['name']

print(f"Found: {microsoft_name} (ID: {microsoft_id})")

Found: Microsoft Corp. (ID: 228D42)


In [6]:
# Example: Search with Microsoft Filter (using Search API)
# Note: This requires the Search API endpoint, which is shown here for completeness
# In a real workflow, you would use this entity ID with the Search API

SEARCH_ENDPOINT = f"{API_BASE_URL}/v1/search"
TEXT = "Global semiconductor shortage impacts"
START_DATE = "2021-01-01"
END_DATE = "2021-12-30"

search_query = {
    "query": {
        "text": TEXT,
        "auto_enrich_filters": False,
        "filters": {
            "timestamp": {
                "start": f"{START_DATE}T00:00:00Z",
                "end": f"{END_DATE}T23:59:59Z"
            },
            "entity": {"any_of": [microsoft_id]}
        },
        "ranking_params": {"freshness_boost": 0},
        "max_chunks": 50
    }
}

response = session.post(SEARCH_ENDPOINT, json=search_query)
search_results = response.json()

print(f"Found {len(search_results.get('results', []))} documents")

Found 48 documents


In [7]:
# Extract Entity IDs from First Chunk
if search_results.get('results'):
    first_chunk = search_results['results'][0]['chunks'][0]
    
    entity_ids = [
        d['id'] for d in first_chunk['detections'] 
        if d['type'] == 'entity'
    ]
    
    print(f"Found {len(entity_ids)} entity IDs in first chunk")
    print(f"Sample IDs: {entity_ids[:5]}")
else:
    print("No search results available")
    entity_ids = []

Found 8 entity IDs in first chunk
Sample IDs: ['60E849', '5442E4', '30DCCC', '9CC31E', '9CC31E']


In [8]:
# Resolve Entity IDs to Names
if entity_ids:
    response = session.post(KG_ENTITIES_ENDPOINT, json={"values": entity_ids})
    results = response.json()['results']
    
    # Results is a dict: entity_id -> entity_object
    entity_map = {entity_id: entity['name'] for entity_id, entity in results.items()}
    
    print(f"Resolved {len(entity_map)} entities:")
    for entity_id, name in list(entity_map.items())[:10]:
        print(f"  {entity_id}: {name}")
else:
    print("No entity IDs to resolve")

Resolved 7 entities:
  228D42: Microsoft Corp.
  5442E4: Industry
  FA1880: Data Center
  60E849: Laptops
  9CC31E: Demand
  F20FAB: Restriction
  30DCCC: Televisions


## 4. Sources: find source IDs and metadata

Finally, we look at **sources**—how to find source IDs by name and filter by rank, category, or country. The Search API returns documents with a `source` object (id, name, rank). To find source IDs by name or to filter sources by quality/category/country, use the Knowledge Graph **Sources** endpoint. Handy when you want to restrict results to certain publishers or align with `source_boost` strategies.

**Endpoint:** [Find sources](https://docs.bigdata.com/api-reference/knowledge-graph/find-sources)

**Available filters**
- `ranks`: e.g. `["RANK_1", "RANK_2"]` (RANK_1 = highest quality)
- `categories`: `["news", "transcripts", "research", "podcasts", "filings", "expert_interviews"]`
- `countries`: ISO 3166-1 alpha-2 codes (e.g. `["US", "GB"]`)
- `packages`: data packages (e.g. `["sec_filings"]`)


In [9]:
# Example: Find Reuters sources with high quality rank
source_query = {
    "query": "Reuters",
    "ranks": ["RANK_1", "RANK_2"],
    "categories": ["news"]
}

response = session.post(KG_SOURCES_ENDPOINT, json=source_query)
sources = response.json()['results']

# Get first source (most relevant)
if sources:
    reuters_source_id = sources[0]['id']
    reuters_source_name = sources[0]['name']
    reuters_source_rank = sources[0]['rank']
    print(f"Reuters source ID: {reuters_source_id}, name: {reuters_source_name}, rank: {reuters_source_rank}")
else:
    print("No sources found")

Reuters source ID: 751371, name: Reuters, rank: RANK_1


## Operational next steps

**What we covered**
- Company lookup by name to get entity IDs for Search/Volume filters
- Resolving entity IDs to names and details (e.g. from search chunk `detections`)
- End-to-end workflow: company → Search with entity filter → resolve IDs from chunks
- Finding sources by name and filtering by rank, category, or country

**Next steps**
- **Search API** ([Search_API](../Search_API/)) — semantic search with entity, time, and sentiment filters
- **Volume API** ([Volume_API](../Volume_API/)) — document and chunk volume over time
- **CoMentions API** ([CoMentions_API](../CoMentions_API/)) — entity co-occurrence; use Knowledge Graph to resolve IDs to names
- **Full API reference:** [docs.bigdata.com](https://docs.bigdata.com)

You can experiment by changing the company query (e.g. `"Microsoft"`, `"Tesla"`) or the source query (e.g. `"Reuters"`) and re-running the notebook to explore different lookups.
